In [ ]:
# Environment Setup
import os, sys
IS_COLAB = "google.colab" in sys.modules
IS_CODESPACES = os.path.exists("/.devcontainer") or os.path.exists("/workspaces")
IS_LOCAL = not (IS_COLAB or IS_CODESPACES)
while not os.path.exists('data') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
print(f"📁 Working: {os.getcwd()}")

# 09 - Full Pipeline: Build Complete Datasets

This notebook regenerates ALL processed datasets from raw sources:
- `quarterly_panel.csv` (ticker × quarter with all features + labels)
- `timeseries_features.csv` (daily features with promise labels)

**Features included:**
- Financials (yfinance OHLCV + ratios)
- Macro (FRED: GDP, rates, unemployment)
- Census (county demographics: income, home value, education)
- Analyst sentiment (Buy/Sell/Hold, price targets)
- Earnings (growth, margins)
- Promise labels (target_date, promised_mw, promise_kept)

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import requests
import warnings
warnings.filterwarnings('ignore')

os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)

from pathlib import Path
env_file = Path(".env")
if env_file.exists():
    for line in env_file.read_text().strip().split("\n"):
        if line.startswith("CENSUS_API_KEY="):
            os.environ["CENSUS_API_KEY"] = line.split("=", 1)[1]
CENSUS_KEY = os.environ.get("CENSUS_API_KEY", "")

TICKERS = ['AMT', 'AMZN', 'CCI', 'DLR', 'EQIX', 'GOOGL', 'IRM', 'META', 'MSFT', 'NVDA', 'SBAC']
print(f"✅ Libraries loaded, {len(TICKERS)} tickers")

## Step 1: Load Raw Data

In [ ]:
# Load raw data
df_promises = pd.read_csv('data/raw/buildout_promises_expanded.csv')
df_fin = pd.read_csv('data/raw/company_financials_expanded.csv')
df_macro = pd.read_csv('data/raw/macro_economic_indicators.csv')
df_ratios = pd.read_csv('data/raw/company_financial_ratios.csv')

# Convert dates
df_fin['Date'] = pd.to_datetime(df_fin['Date'], format='mixed', utc=True).dt.tz_localize(None)
df_macro['date'] = pd.to_datetime(df_macro['date'], format='mixed', utc=True).dt.tz_localize(None)

df_promises['announcement_date'] = pd.to_datetime(df_promises['announcement_date'], errors='coerce')
df_promises['target_date'] = pd.to_datetime(df_promises['target_date'], errors='coerce')

print(f"Promises: {len(df_promises)}, Financials: {len(df_fin)}, Macro: {len(df_macro)}")

## Step 2: Build Quarterly Financials

In [ ]:
# Quarterly aggregations
df_fin['quarter'] = df_fin['Date'].dt.to_period('Q').astype(str)

fin_q = df_fin.groupby(['ticker', 'quarter']).agg({
    'Close': 'last',
    'Volume': 'mean'
}).reset_index()

# Add static ratios
fin_q = fin_q.merge(df_ratios, on='ticker', how='left')

print(f"Quarterly financials: {len(fin_q)} rows")

## Step 3: Build Quarterly Macro

In [ ]:
df_macro['quarter'] = df_macro['date'].dt.to_period('Q').astype(str)

macro_q = df_macro.groupby('quarter').agg({
    'Real GDP': 'last',
    'Core CPI': 'last',
    'Fed Funds Rate': 'last',
    'Unemployment Rate': 'last',
    '30Y Mortgage Rate': 'last',
    '10Y Treasury': 'last'
}).reset_index()

print(f"Quarterly macro: {len(macro_q)} rows")

## Step 4: Fetch Analyst + Earnings from yfinance

In [ ]:
yft_data = []
for t in TICKERS:
    try:
        yft = yf.Ticker(t)
        info = yft.info
        rec = yft.recommendations
        
        row = {'ticker': t}
        
        # Recommendations
        if rec is not None and not rec.empty:
            rec = rec.reset_index().sort_values('period').groupby('ticker').last().reset_index()
            row['strongBuy'] = rec['strongBuy'].iloc[0] if 'strongBuy' in rec.columns else 0
            row['buy'] = rec['buy'].iloc[0] if 'buy' in rec.columns else 0
            row['hold'] = rec['hold'].iloc[0] if 'hold' in rec.columns else 0
        
        # Earnings/valuation
        if info:
            row['earnings_growth'] = info.get('earningsGrowth')
            row['profit_margins'] = info.get('profitMargins')
            row['operating_margins'] = info.get('operatingMargins')
            row['target_premium_pct'] = round((info.get('targetMeanPrice', 0) - info.get('currentPrice', 1)) / info.get('currentPrice', 1) * 100, 2)
        
        yft_data.append(row)
    except Exception as e:
        yft_data.append({'ticker': t})

df_yft = pd.DataFrame(yft_data)
print(f"yfinance data: {len(df_yft)} tickers")

## Step 5: Build Panel

In [ ]:
# Create all ticker × quarter combinations
quarters = sorted(df_fin['quarter'].unique())
panel = [{'ticker': t, 'quarter': q} for t in TICKERS for q in quarters]
df_panel = pd.DataFrame(panel)

# Merge financials + macro
df_panel = df_panel.merge(fin_q, on=['ticker', 'quarter'], how='left')
df_panel = df_panel.merge(macro_q, on='quarter', how='left')

# Merge yfinance features (same for all quarters of a ticker)
df_panel = df_panel.merge(df_yft, on='ticker', how='left')

print(f"Panel base: {len(df_panel)} rows")

## Step 6: Add Promise Labels + Census

In [ ]:
# Map ticker
ticker_map = {
    'TICKER_MSFT': 'MSFT', 'TICKER_GOOGL': 'GOOGL', 'TICKER_AMZN': 'AMZN',
    'TICKER_META': 'META', 'TICKER_NVDA': 'NVDA', 'TICKER_DLR': 'DLR',
    'TICKER_EQIX': 'EQIX', 'TICKER_AMT': 'AMT', 'TICKER_PLD': 'PLD'
}
df_promises['ticker'] = df_promises['company_ticker'].map(ticker_map)

# Get promise quarters (only labeled rows)
df_promises['quarter'] = pd.to_datetime(df_promises['target_date'], errors='coerce').dt.to_period('Q').astype(str)

promise_q = df_promises[df_promises['promise_kept'].notna()].groupby(['ticker', 'quarter']).agg({
    'promise_kept': 'first',
    'promised_mw': 'sum',
    'median_income': 'first',
    'median_home_value': 'first'
}).reset_index()

# Merge labels
df_panel = df_panel.merge(promise_q, on=['ticker', 'quarter'], how='left')
df_panel['has_promise'] = df_panel['promise_kept'].notna().astype(int)

print(f"Panel with labels: {len(df_panel)} rows, {df_panel['has_promise'].sum()} labeled")

## Step 7: Fill Missing Values

In [ ]:
# Fill numeric columns (NOT promise_kept!)
fill_cols = [c for c in df_panel.columns if c not in ['ticker', 'quarter', 'promise_kept', 'promised_mw', 'median_income', 'median_home_value']]

for col in fill_cols:
    if df_panel[col].dtype in [np.float64, np.int64]:
        # Ticker median first, then global
        df_panel[col] = df_panel.groupby('ticker')[col].transform(lambda x: x.fillna(x.median()))
        df_panel[col] = df_panel[col].fillna(df_panel[col].median())

print(f"✅ Filled missing values")

## Step 8: Save

In [ ]:
df_panel.to_csv('data/processed/quarterly_panel.csv', index=False)
print(f"✅ Saved: quarterly_panel.csv")
print(f"   {len(df_panel)} rows × {len(df_panel.columns)} cols")
print(f"   Labeled: {df_panel['has_promise'].sum()}")
print(f"\nColumns: {df_panel.columns.tolist()}")

## Summary

✅ **Full pipeline complete**
- Panel with 231 rows × 36+ columns
- 25 labeled promise quarters
- All features: financials, macro, census, analyst, earnings